# Comparioson 

In [1]:
import sys
sys.path.insert(0, '..')
import torch
import pandas as pd
from gensim.models import KeyedVectors
from src.models import build_model
from src.data import load_data
from src.evaluate import run_all_evaluations

In [2]:
embeddings = KeyedVectors.load_word2vec_format(
    '../gensim-data/glove-wiki-gigaword-300/glove-wiki-gigaword-300.gz'
)
print(f'Vocab size: {len(embeddings.key_to_index)}')

Vocab size: 400000


In [3]:
from src.data import load_data

_, _, gender_clean, royalty_clean = load_data(embeddings)


--------------------------------------------------
  GENDER PAIRS
--------------------------------------------------
  candidates: 100 | valid ones: 100 | discarded: 0

--------------------------------------------------
  ROYALTY PAIRS
--------------------------------------------------
  candidates: 106 | valid ones: 106 | discarded: 0

  analysis: GENDER
  cos mean=0.4609 std=0.1898 | threshold 0.3: 83/100

  analysis: STATUS
  cos mean=0.2943 std=0.1196 | threshold 0.3: 51/106

  Cosine between axes: 0.3172

  Train: 106 | Val: 28


In [4]:
EXPERIMENTS = {
    'linear':          '../results/linear/model.pt',
    'linear_bias':     '../results/linear_bias/model.pt',
    'film_gamma':      '../results/film_gamma/model.pt',
    'film_gamma_beta': '../results/film_gamma_beta/model.pt',
}

rows = []
for name, path in EXPERIMENTS.items():
    model = build_model(name)
    model.load_state_dict(torch.load(path, map_location='cpu'))
    model.eval()
    print(f'\n{"="*40}\n  {name}\n{"="*40}')
    metrics = run_all_evaluations(model, embeddings, gender_pairs=gender_clean, royalty_pairs=royalty_clean)
    rows.append({'model': name, **metrics})

df = pd.DataFrame(rows).set_index('model')
df


  linear
Analigies:
  king - man + woman → queen  |  cos=0.9887  dist=0.2921
  prince - boy + girl → princess  |  cos=0.9819  dist=0.9253
  husband - man + woman → wife  |  cos=0.9585  dist=1.1758
  uncle - man + woman → aunt  |  cos=0.9826  dist=0.2453
  father - man + woman → mother  |  cos=0.9999  dist=0.0984
  brother - man + woman → sister  |  cos=0.9930  dist=0.3289
  king - prince + princess → queen  |  cos=0.9890  dist=0.5588
  grandfather - father + mother → grandmother  |  cos=0.9992  dist=0.1803
  actor - man + woman → actress  |  cos=0.9900  dist=0.6933
  he - man + woman → she  |  cos=0.9937  dist=0.2707
  analogy_cos=0.9877 | analogy_dist=0.4769

Quality of Axes
  gender_purity=0.9937 | status_purity=0.9764
  gender_leakage=0.0325 | status_leakage=0.0742

  linear_bias
Analigies:
  king - man + woman → queen  |  cos=0.9879  dist=0.2424
  prince - boy + girl → princess  |  cos=0.9933  dist=0.2513
  husband - man + woman → wife  |  cos=0.9972  dist=0.8001
  uncle - man + w

,analogy_cos,analogy_dist,gender_purity,status_purity,gender_leakage,status_leakage
model,,,,,,
linear,0.987658,0.476874,0.993738,0.976438,0.032499,0.074152
linear_bias,0.987094,0.365947,0.987627,0.979106,0.089977,0.141582
film_gamma,0.875671,0.508571,0.511463,0.156251,0.369003,0.551811
film_gamma_beta,0.604765,0.578821,-0.145886,0.168757,0.606466,0.582397
